In this Jupyter notebook, I define the functions necessary for simulating the quantum circuit of the tranverse field Ising model. I will start with methods used in this IBM tutorial ([https://quantum.cloud.ibm.com/docs/en/tutorials/transverse-field-ising-model](https://quantum.cloud.ibm.com/docs/en/tutorials/transverse-field-ising-model)) which claims to be an efficient way to Trotterize the TFI model.

In [3]:
import numpy as np
from numpy.linalg import matrix_power
from numpy.linalg import eigh
from scipy.special import factorial
from scipy.sparse.linalg import expm_multiply
import functools as ft


from itertools import product

from qiskit import QuantumCircuit, transpiler, QuantumRegister, ClassicalRegister, AncillaRegister, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.circuit.classical import expr
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.synthesis import SuzukiTrotter

from qiskit_aer import AerSimulator


Generate the Hamiltonian operator $H = - \sum_i J Z_i Z_{i+1} - h X_i$ for an $N$ qubit chain.

In [65]:
def ham_couple(qc, J):
    ''' Generates the site-site coupling part of the Hamiltonian and returns a SparsePauliOp'''
    num_qubits = qc.num_qubits - qc.num_ancillas
    pauli_string = 'ZZ' + 'I' * (num_qubits -2)
    
    ops_list = [(pauli_string[shift:] + pauli_string[:shift], -1.0 *J) for shift in range(num_qubits)]
    ham = SparsePauliOp.from_list(ops_list)
    return ham

def ham_zeeman(qc, h):
    ''' Generates the coupling to the external field and returns a SparsePauliOp'''
    num_qubits = qc.num_qubits - qc.num_ancillas
    pauli_string = 'X' + 'I' * (num_qubits -1)
    
    ops_list = [(pauli_string[shift:] + pauli_string[:shift] , -1.0 *h) for shift in range(num_qubits)]
    ham = SparsePauliOp.from_list(ops_list)
    return ham

def Hamiltonian(qc, J, h):
    '''Generates the full Hamiltonian as a SparsePauliOp'''
    return ham_couple(qc, J) + ham_zeeman(qc, h)

qc = circuit_initialize(4, 2, 2)
ham_zeeman(qc,1)


SparsePauliOp(['XIII', 'IIIX', 'IIXI', 'IXII'],
              coeffs=[-1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j])

Initialize the Quantum circuit with a qubit, ancilla, and classical register

In [15]:

def circuit_initialize(num_qubits, num_ancilla, num_classical):
    '''Builds a circuit appropriate for a num_qubit Ising chain.'''
    
    state_qubits = QuantumRegister(num_qubits, "s")
    ancilla_qubits = AncillaRegister(num_ancilla, "a")
    classical_bits = ClassicalRegister(num_classical, "c")
    qc = QuantumCircuit(state_qubits , ancilla_qubits, classical_bits)
    return qc

Time evolution is to be Trotterized by a second order operation: $\exp(A + B) \approx \exp(A/2) \exp(B)\exp(A/2)$. The time step will be controlled by the delta parameter which must be optimized to ensure time evolution is accurate while maintaining a reasonable number of gates. 

In [73]:
def trotter_evolve(qc, time_end, delta, J, h):
    '''appends to a quantum circuit the time evolution by Trotterized exponential Hamiltonian '''

    num_qubits = qc.num_qubits - qc.num_ancillas
    for t in np.arange(0, time_end, delta):
        for i in range(num_qubits):
            qc.rx(-1.0 * h * delta/2, i)
            qc.rzz(-1.0 * J * delta, i, (i+1) % num_qubits)
            qc.rx(-1.0 * h * delta/2, i)


def trotter_evolve_PauliEvolutionGate(qc, time_end, delta, J, h, order = 2, reps = 2):
    """Uses built in function to Trotterize (for comparison with manual Trotterization)"""

    num_qubits = qc.num_qubits - qc.num_ancillas
    num_steps = int(time_end// delta)

    for dt in range(num_steps):
        qc.append(PauliEvolutionGate(Hamiltonian(qc, J, h), delta , 
                                     synthesis=SuzukiTrotter(order , reps ), label = "$\\exp\\{- i H (t)\\}") , range(num_qubits))



        

Construct measurements: 1) spin Z on a single site, 2) spin X on all even sites

In [90]:
def spin_Z_measure(qc, register_bit):
    '''measures spin in z basis of site'''
    for reg in qc.qregs:  #Define the ancilla register
        if reg.name == "a":
            ancilla_qubits = reg
            break
    
    for reg in qc.cregs:    #Define the classical register
        if reg.name == "c":
            classical_bits = reg
            break

    qc.measure(0, classical_bits[register_bit])

def spin_X_even_measure(qc, register_bit):
    '''measures the total of the spins in x basis of just even sites'''
    for reg in qc.qregs:  #Define the ancilla register
        if reg.name == "a":
            ancilla_qubits = reg
            break
    
    for reg in qc.cregs:    #Define the classical register
        if reg.name == "c":
            classical_bits = reg
            break

    num_qubits = qc.num_qubits - qc.num_ancillas

    for i in range(0, num_qubits, 2):
        qc.h(i)
        qc.measure(i, classical_bits[i//2+ register_bit * num_qubits//2])
        qc.h(i)


Now construct the whole circuit.

In [95]:
def ising_correlation_circuit(qc,  J, h, time_end, delta, measure ):
    '''constructs circuit which measures 'measure' at the start, and then at the end and reads into clbits'''
    
    if measure == 'Z':
        spin_Z_measure(qc, 0)
    elif measure == 'X_even':
        spin_X_even_measure(qc, 0)
    else:
        raise ValueError("measure must be either 'Z' or 'X_even'")
    trotter_evolve(qc, time_end, delta, J, h)
    if measure == 'Z':
        spin_Z_measure(qc, 1)
    elif measure == 'X_even':
        spin_X_even_measure(qc, 1)
    else:
        raise ValueError("measure must be either 'Z' or 'X_even'")
    

We first need to run on a simulator to optimize the Trotter evolution delta parameter, and to make sure the LG bound is violated. We will first choose to measure the single site $Z$ operator. It's worth noting that the Abboud, Bradlyn paper claims that at any finite time, and for any finite $h$ in the ordered phase $J > h$ that the LG bound is immediately violated. Because we don't have access to the exact ground state for any $h, J \neq 0$ without constructing a variational ansatz, we will approximate the ground states in the limit with either a polarized set of qubits in the z or x basis (for the $J/h \gg 1$ and $h/J \gg 1$ limits respectively). After confirming our calculations, we will construct the ansatz wavefunction. 

In [108]:
#First the h/J >> 1 limit

C = []
for j in range(1,3):

    chain_len = 4
    qc = circuit_initialize(chain_len, 0, 2)
    num_shots = 10000

    #ground state is all spins in the positive x direction, so we can prepare it by applying H to all qubits
    for i in range(chain_len):
        qc.h(i)

    ising_correlation_circuit(qc, J = 1, h = 10, time_end = 0.5 * j, delta = 0.05, measure = 'Z')
    backend = AerSimulator()
    job = backend.run(qc, shots = num_shots)
    counts = job.result().get_counts()
    C.append((counts["00"]+ counts["11"] - counts["01"] - counts["10"]) / num_shots)


    #Calculate the correlation from the counts

2 *C[0] - C[1]

1.1568

In [104]:
(counts["00"]+ counts["11"] - counts["01"] - counts["10"]) / num_shots

-0.6466